In [110]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np #专门做数字计算的工具包，全名 Numerical Python

path='/content/drive/MyDrive/ai_project/adults-media-literacy-2025-respondent-level-data.csv'
df=pd.read_csv(path, na_values=' ')

sample=df[(df['MODULEA']==1)&(df['INTUSED']==1)].copy()

#复印一份独立的新表，只包含有效样本，下面的清洗都是在sample上增加改动改它不会影响 df

sample.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(3515, 566)

In [111]:
sample['Y1']=sample['A6'].map({1:3,2:2,3:1,4:0}) #在 sample 里新建一列，名字叫 Y1。左边写一个还不存在的列名，pandas 就会新建它

                                                 #{} dictionary（字典）, 查key 输出value--构造 Y1（A6 重新编码）

                                                 #.map()将指定的函数依次作用于一个或多个可迭代对象（如列表、字典等）的每个元素上，并返回一个包含结果的迭代器

                                                 # .map() Series列 method 一列里的每一个值，把它当成 key 去字典里查，然后用查到的 value 替换它。

                                                 # 字典里没有5, .map() 查不到，就自动变成 NaN

sample['Y1'].value_counts(dropna=False).sort_index()#让 value_counts() 把 NaN 也数出来（默认是不数的--dropna=True)

                                                    #value_counts() 默认按人数从多到少排。.sort_index() 是按行名（这里就是 0、1、2、3）从小到大排

                                                    #按值从小到大排（0、1、2、3），而不是按人数多少排，方便看

,count
Y1,
0.0,525
1.0,1156
2.0,893
3.0,604
NaN,337


In [112]:
#.map()底层逻辑
#def change_value(x):
#    if x == 1: return 3
#    elif x == 2: return 2
#    elif x == 3: return 1
#    elif x == 4: return 0
#    else: return None  # 其他找不到的变空值

In [113]:
sample['seen_summary']=(sample['A6']!=5).astype(int)#你的 A6 不是 5 吗？不是 5 → True（见过），是 5 → False（没见过）

                                                   #A6如果是Na!=5-True,没被提问会被当作见过，但sample里没有NaN,df才有

                                                   #astype = “as type”，换成某种类型。int 是整数。True 变成 1，False 变成 0

                                                   #1/0 以后放进模型、算比例更方便（比如直接 .mean() 就是"见过的人占多少"）
sample['seen_summary'].value_counts()

,count
seen_summary,
1,3178
0,337


In [114]:
sample['income_dk']=(sample['E9']==8).astype(int)
sample['income_dk'].value_counts()

,count
income_dk,
0,2788
1,727


In [115]:
(sample['SUMAI']==3).value_counts()


,count
SUMAI,
False,3274
True,241


In [116]:
sample['A4_c']=sample['A4'].fillna(0)
sample['A4_c'].value_counts(dropna=False).sort_index()

,count
A4_c,
0.0,241
1.0,246
2.0,876
3.0,1841
4.0,311


In [117]:
tab=pd.crosstab(sample['SUMAI'],sample['A4_c'].fillna(0))
tab

A4_c,0.0,1.0,2.0,3.0,4.0
SUMAI,,,,,
1.0,0,214,713,1014,119
2.0,0,32,163,827,192
3.0,241,0,0,0,0


##Check: A4_c = 0 only for respondents unaware of AI (SUMAI = 3)

In [118]:
sample['A5_c']=sample['A5'].fillna(0)
sample['A5_c'].value_counts(dropna=False).sort_index()

,count
A5_c,
0.0,241
1.0,348
2.0,1144
3.0,935
4.0,573
5.0,214
6.0,60


In [119]:
tab1=pd.crosstab(sample['SUMAI'],sample['A5_c'].fillna(0))
tab1

A5_c,0.0,1.0,2.0,3.0,4.0,5.0,6.0
SUMAI,,,,,,,
1.0,0,289,864,545,284,60,18
2.0,0,59,280,390,289,154,42
3.0,241,0,0,0,0,0,0


In [120]:
sample.shape

(3515, 571)

In [121]:
out_path = '/content/drive/MyDrive/ai_project/sample_clean_v1.csv'#out是输出，和path区分，v1第一版
sample.to_csv(out_path, index=False)
#不要把行名存进去。不写的话，pandas 会把行名（行号 3、7、8……）当成多出来的一列存进文件，下次读进来就会多一列没名字的东西

In [122]:
check = pd.read_csv(out_path)#检查
check.shape

(3515, 571)

## Cleaning part 2 (Y variable)

##下一步ai使用广度

In [123]:
a3_cols=['A3A','A3B','A3C','A3D','A3E','A3F','A3G','A3H']

In [124]:
sample[a3_cols].head(10)

,A3A,A3B,A3C,A3D,A3E,A3F,A3G,A3H
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
23,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Y2 是怎么算出来的（4 个人举例）

**① `sample[a3_cols]`：从大表里切出 8 列**

| 行名 | SUMAI | A3A | A3B | A3C | A3D | A3E | A3F | A3G | A3H | 说明 |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | 1 | 1 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 用过 AI，选了 2 项 |
| 7 | 2 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 没用过（Ofcom 填的 0） |
| 9 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 用过，一项没选（那 36 人之一） |
| 13 | 3 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 没听说过 AI（Ofcom 填的 0） |

In [125]:
sample[a3_cols].sum(axis=1)#.sum()和.sum(axis=0)都是一列往下相加，算得是每个活动有多少人选,
                           #.sum(axis=1)是一行相加，每个人选了几种-Y2预测ai广度用

,0
1,0.0
7,0.0
9,0.0
13,0.0
15,1.0
...,...
7521,2.0
7523,0.0
7524,1.0
7525,1.0


**② `.sum(axis=1)`：每行横着加，8 列压成 1 列**

In [126]:
sample[a3_cols].sum(axis=1).value_counts().sort_index()#ofcom数据把sumai=2，3没用过ai也标0，包含一部分没用过ai要当空值，一部分填a3i=其他用途，a3j不知道，这两个是应该保留的
                                                       #上个列表算出来是每行=每个人8个算了几个
                                                       #value_counts 每一个唯一值出现次数，1491就是一共有1491人选0
                                                       #sort按标签，不按人数多少

,count
0.0,1491
1.0,555
2.0,484
3.0,395
4.0,249
5.0,163
6.0,108
7.0,41
8.0,29


In [127]:
((sample['SUMAI']==2)|(sample['SUMAI']==3)).value_counts()#1455就是没用过ai的人，应该要当空值处理
#|或者，&并且
#(sample['SUMAI'] != 1).value_counts()同样效果
#sample['SUMAI'].isin([2, 3]).value_counts()，.isin()："是不是在这个 list 里面"：

,count
SUMAI,
False,2060
True,1455


**③ `no_aiusers = (SUMAI==2) | (SUMAI==3)`：每个人一张 True/False 牌**

| 行名 | SUMAI | no_aiusers |
|---|---|---|
| 1 | 1 | False |
| 7 | 2 | **True** |
| 9 | 1 | False |
| 13 | 3 | **True** |

In [128]:
no_aiusers=((sample['SUMAI']==2)|(sample['SUMAI']==3))

In [129]:
zero_users = sample[(sample['SUMAI'] == 1) & (sample[a3_cols].sum(axis=1) == 0)]#1491-1455=36是满足用过ai，ai用处填0的有36人，
zero_users.shape

(36, 571)

In [130]:
pd.crosstab(zero_users['A3I'],zero_users['A3J'])#填a3j=17人是不知道，a3i=19人是其他用途，不再前面8种里

A3J,0.0,1.0
A3I,,
0.0,0,17
1.0,19,0


In [131]:
sample['Y2']=(sample[a3_cols].sum(axis=1))
sample.loc[no_aiusers,'Y2']=np.nan
sample['Y2'].value_counts(dropna=False).sort_index()

,count
Y2,
0.0,36
1.0,555
2.0,484
3.0,395
4.0,249
5.0,163
6.0,108
7.0,41
8.0,29


**④ `sample['Y2'] = ...`：贴回大表，叫 Y2**

| 行名 | SUMAI | Y2 | 说明 |
|---|---|---|---|
| 1 | 1 | 2 | |
| 7 | 2 | 0 | 假的 0 |
| 9 | 1 | 0 | 真的 0 |
| 13 | 3 | 0 | 假的 0 |

**⑤ `sample.loc[no_aiusers, 'Y2'] = np.nan`：True 的行，把 Y2 改成空**

| 行名 | no_aiusers | Y2 改之前 | Y2 改之后 |
|---|---|---|---|
| 1 | False | 2 | 2（不动） |
| 7 | True | 0 | **NaN** |
| 9 | False | 0 | 0（真的 0 保留） |
| 13 | True | 0 | **NaN** |


##下一步，x2其中（上网活动数）

In [132]:
activity_cols = ['Q13A', 'Q13B', 'Q13C', 'Q13D', 'Q13E', 'Q13F', 'Q13G', 'Q13H', 'Q13I', 'Q13J', 'Q13K', 'Q13L', 'Q13M']

# 计算每个人在 Q13A–Q13M 中选择了多少个活动，并存入新列 activity_count
sample['activity_count'] = sample[activity_cols].sum(axis=1)

# 检查新列的分布
sample['activity_count'].value_counts(dropna=False).sort_index()

,count
activity_count,
0,54
1,74
2,166
3,206
4,291
5,317
6,407
7,386
8,375


In [133]:
pd.crosstab(sample['BREADTH'],sample['activity_count'])

activity_count,0,1,2,3,4,5,6,7,8,9,10,11,12,13
BREADTH,,,,,,,,,,,,,,
1,54,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,74,166,206,291,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,317,407,386,375,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,371,308,239,185,136


In [134]:
v2_path = '/content/drive/MyDrive/ai_project/sample_clean_v2.csv'
sample.to_csv(v2_path, index=False)

check = pd.read_csv(v2_path)
check.shape

(3515, 573)

## Cleaning part 2 (X variable)

##下一步设备数量

In [135]:
device_cols=['Q10A','Q10B','Q10C','Q10D','Q10E','Q10F']#设备数量，只有一种的919人，2种设备1030人
sample['device_counts']=sample[device_cols].sum(axis=1)
sample['device_counts'].value_counts(dropna=False).sort_index()

,count
device_counts,
1.0,919
2.0,1030
3.0,865
4.0,463
5.0,234
6.0,4


##下一步社交平台数量

In [136]:
social_cols=['Q16A','Q16B','Q16C','Q16D','Q16E','Q16F','Q16G','Q16H','Q16I','Q16J','Q16K','Q16L']
sample['social_counts']=sample[social_cols].sum(axis=1)
sample['social_counts'].value_counts(dropna=False).sort_index()

,count
social_counts,
0,378
1,269
2,544
3,488
4,492
5,394
6,305
7,220
8,149


## X 变量整理：小类别处理规则

### 为什么要处理小类别？
- 只对**类别变量**有影响（模型会给每一类单独估计一个效果）
- **数字变量**（activity_count、device_counts、social_counts）不受影响，比如设备数 = 6 只有 4 人也没关系
- 小类别的三个问题：
  1. 人太少，估计不可靠
  2. 80/20 划分时，这一类可能只出现在训练集或测试集里
  3. 结果没法解释（置信区间很宽）

### 规则
| 规则 | 内容 |
|---|---|
| 门槛 | 类别变量里少于 **30 人**（约 1%）的类别要处理 |
| 优先 | 能并进相近类别的就合并 |
| 其次 | 能和其他小类凑成"其他/未知"、且凑够 30 人的，合成一类 |
| 最后 | 实在凑不够的，删掉这些人，并在报告里说明 |

### 已决定
| 变量 | 情况 | 处理 |
|---|---|---|
| S9 工作 | 不愿说 10 + 其他 53 | 合并为"其他/不愿说" = 63 人 |
| S4 性别 | 非二元 9 + 不愿说 1 = 10 人 | 删掉（合并后仍不够 30） |
| SEG 社会阶层 | 不知道 9 人 | 删掉 |

- S4 和 SEG 共删 **19 人**（无重叠，占 0.5%），这 19 人的 A6 分布没有明显特点
- **最后一步再删**（存 v3 之前），这样中间检查时人数还能和之前对照
- 删人后要更新 PLAN 里的样本数
- Limitation：删掉的人包括 9 位非二元受访者，结论不能推广到这个群体

### 还没做
- E5 学历：15 个选项，要合并成几档
- S1 上网时间：不知道 38；编码 1 有 18 人，说明书没写是什么
- Q12A–D 上网自信：4 题要不要合成一个分数；不知道 7–26 人
- A7 尝试新网站：上个月没上网 47，不知道 91

In [137]:
sample['edu_c']=sample['E5'].map({1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:3,9:3,10:4,11:4,12:5,13:6,14:7,15:7})
#由于职业类和学术类比如GCSE3-4，初级职业资格5，在学历上都是对应英国同一个等级level 1-2 初中学历，6高中文凭和7高级职业文凭，都对应level 3
#因此把相同学历编码都改为同一个
sample['edu_c'].value_counts(dropna=False).sort_index()

,count
edu_c,
0,231
1,988
2,596
3,326
4,775
5,514
6,47
7,38


## E5 学历：15 个选项合并成 8 类

### 背景：英国的学历 Level（对应中国的大致叫法）
英国把读书拿的学历和职业培训拿的资格放在同一套 Level 里。下面的中国叫法只是**大致对应**，方便理解。

| Level | 英国学术类 | 英国职业类 | 中国大致对应 |
|---|---|---|---|
| 1–2 | GCSE | Level 1–2 职业资格 / 初级学徒 | 初中毕业（中考水平）/ 初级职业培训 |
| 3 | A-level / IB | Level 3 职业资格 / 高级学徒 | 高中（高考水平）/ 中专、职高、技校 |
| 4–5 | 高等教育文凭（HNC / HND） | Level 4–5 职业资格 / 高等学徒 | **大专（专科）/ 高职** |
| 6 | 本科学位 | Level 6 / 学位学徒 | 本科 / 职业本科 |
| 7–8 | 硕士、博士 | | 研究生 |

### 原始 15 个选项（样本 3,515 人）
| 编码 | 内容 | 中国大致对应 | 人数 |
|---|---|---|---|
| 1 | 没有任何正式学历 | 没有学历 | 179 |
| 2 | 入门级（如 ESOL 英语课） | 低于初中 | 52 |
| 3 | GCSE，不含数学和英语 | 初中 | 346 |
| 4 | GCSE，含数学或英语 | 初中 | 475 |
| 5 | Level 1–2 职业资格 | 初级职业培训 | 167 |
| 6 | A-level / IB | 高中 | 399 |
| 7 | Level 3 职业资格 | 中专 / 职高 | 197 |
| 8 | HNC / HND / BTEC Higher | 大专（专科） | 225 |
| 9 | Level 4–5 职业资格 | 高职 / 大专 | 101 |
| 10 | 本科学位 | 本科 | 725 |
| 11 | Level 6 职业资格 / 学位学徒 | 职业本科 | 50 |
| 12 | 硕士、博士 | 研究生 | 514 |
| 13 | 还在读书 | 不知道目前水平 | 47 |
| 14 | 不知道 | | 11 |
| 15 | 不愿说 | | 27 |

### 合并的思路
1. 编码 3–12：按 Level 合并，同一 Level 的学术类和职业类放一起
2. 编码 1 + 2（没学历 / 入门级，231 人）：低于 Level 1，**单独一类**。如果并进初中那档，会把"没有学历"和"有初中学历"的差别抹掉
   - 注意：样本里的人都上网，所以这 231 人也会用互联网，但他们可能年纪更大、上网做的事更少
3. 编码 13（还在读书，47 人）：没选 1–12，不知道目前水平；够 30 人，**单独一类**
4. 编码 14 + 15（不知道 + 不愿说，11 + 27 = 38 人）：够 30 人，**合成一类**

### 最终分组：新列 `edu_c`
| 新编号 | 名字 | 原编码 | 中国大致对应 | 人数 |
|---|---|---|---|---|
| 0 | 没有学历 / 入门级 | 1, 2 | 低于初中 | 231 |
| 1 | Level 1–2 | 3, 4, 5 | 初中 | 988 |
| 2 | Level 3 | 6, 7 | 高中 / 中专 | 596 |
| 3 | Level 4–5 | 8, 9 | 大专（专科） | 326 |
| 4 | Level 6 | 10, 11 | 本科 | 775 |
| 5 | Level 7–8 | 12 | 研究生 | 514 |
| 6 | 还在读书 | 13 | | 47 |
| 7 | 不知道 / 不愿说 | 14, 15 | | 38 |

- 每类都超过 30 人，加起来 3,515
- 0–5 从低到高；6 和 7 不在这个顺序里，所以 `edu_c` 放进模型时当**类别**，不当数字大小

##下一步S9工作状态

## S9 工作：把"不愿说"并进"其他"

### 原始选项
| 编码 | 内容 | 人数 |
|---|---|---|
| 1 | 全职 | 1,673 |
| 2 | 兼职 | 572 |
| 3 | 失业 | 241 |
| 4 | 学生 | 203 |
| 5 | 全职照顾家庭 | 165 |
| 6 | 退休 | 598 |
| 7 | 其他 | 53 |
| 8 | 不愿说 | 10 |

### 处理
- "不愿说"只有 10 人，少于 30
- 并进"其他"：53 + 10 = **63 人**，够 30
- 新列 `work_c`：编码 8 → 7，其他不变

### `.map()` 和 `.replace()` 的区别
| 方法 | 字典里**没写**的值 | 适合什么情况 |
|---|---|---|
| `.map()` | 变成 **NaN** | 每个值都要换（比如 Y1、edu_c） |
| `.replace()` | **保持原样** | 只改几个，其他不动（比如这里只改 8） |

### 检查
- `value_counts(dropna=False).sort_index()`：只有 1–7，编码 7 = 63 人，没有 NaN

In [138]:
sample['work_c']=sample['S9'].replace({8:7})
sample['work_c'].value_counts(dropna=False).sort_index()

,count
work_c,
1,1673
2,572
3,241
4,203
5,165
6,598
7,63


##下一个：S1 每周上网时间

## S1 每周上网时间：编码 1 并进编码 2

### 原始选项
| 编码 | 内容 | 人数 |
|---|---|---|
| 1 | None（每周 0 小时） | 18 |
| 2 | 最多 2 小时 | 61 |
| 3 | 3–7 小时 | 289 |
| 4 | 8–11 小时 | 340 |
| 5 | 12–15 小时 | 420 |
| 6 | 16–22 小时 | 611 |
| 7 | 23–30 小时 | 701 |
| 8 | 30 小时以上 | 1,037 |
| 9 | 不知道 | 38 |

### 编码 1 是什么
- 说明书写的是 **None**（每周 0 小时）
- 注意：pandas 读 Excel 时会把 "None" 当成空值，所以用代码查说明书时显示成了 nan，要回原文件确认
- 完整数据里，不上网的 504 人全部是编码 1；样本里这 18 人是上网的（INTUSED = 1），可能不是每周都上
- 这 18 人上网活动平均 1.8 种（全样本最少），年龄最大

### 处理
| 编码 | 处理 | 理由 |
|---|---|---|
| 1（18 人） | **并进 2**，合并后叫"每周 0–2 小时"，共 **79** 人 | 少于 30；和编码 2 相邻，意思接近；规则是优先合并 |
| 9 不知道（38 人） | **单独一类**保留 | 够 30 |

### 为什么不删编码 1
- 删人只适合"这些人没什么特别"的情况（比如性别、SEG 那 19 人）
- 这 18 人是**最少上网的人**，删掉会让样本系统性地少了这类人，造成偏差
- 而这类人和研究问题（谁读 AI 摘要）最相关

### 新列 `online_time_c`
- 编码 1 → 2，其他不变
- 检查：只剩 2–9 八类，编码 2 = 79 人，没有 NaN

In [139]:
sample['online_time_c']=sample['S1'].replace({1:2})
sample['online_time_c'].value_counts(dropna=False).sort_index()

,count
online_time_c,
2,79
3,289
4,340
5,420
6,611
7,701
8,1037
9,38


##第 6 步：Q12A–D 上网自信

## Q12A–D 上网自信：四题合成一个分数 `online_conf`

### 四道题
| 题 | 问的是 |
|---|---|
| Q12A | 整体来说，作为互联网用户有多自信 |
| Q12B | 分辨哪些是广告，有多自信 |
| Q12C | 判断网上信息是真是假，有多自信 |
| Q12D | 判断邮件、短信是不是诈骗，有多自信 |

每题：1 非常自信、2 比较自信、3 一般、4 不太自信、5 完全不自信、**6 不知道**

### 为什么合成一个分数
- 分开放进模型：4 题 × 6 类，要估计约 20 个效果，太多
- 四题可能在测**同一件事**：背后都是一个看不见的"上网自信"
  - 就像 4 道数学题都在测"数学能力"：能力强的人，4 题往往都做得好，所以 4 题的得分会一起高、一起低
- 用两个指标检查是不是"同一件事"：相关系数、Cronbach's alpha

---

### 指标一：相关系数（Pearson correlation，r）
**看两题是不是一起高、一起低**，范围 −1 到 1

| r | 意思 |
|---|---|
| 接近 1 | 一题高，另一题几乎一定也高 |
| 0 | 没关系 |
| 接近 −1 | 一题高，另一题反而低 |

**例子：4 个人的两道题（5 = 最自信）**

| 人 | x | y | x − x̄ | y − ȳ | 偏差相乘 | (x − x̄)² | (y − ȳ)² |
|---|---|---|---|---|---|---|---|
| 甲 | 5 | 4 | 2 | 1 | 2 | 4 | 1 |
| 乙 | 4 | 5 | 1 | 2 | 2 | 1 | 4 |
| 丙 | 2 | 2 | −1 | −1 | 1 | 1 | 1 |
| 丁 | 1 | 1 | −2 | −2 | 4 | 4 | 4 |
| **合计** | | | | | **9** | **10** | **10** |

步骤：
1. 算平均：x̄ = 3，ȳ = 3
2. 每个人的偏差 = 自己的分 − 平均
3. **偏差相乘**：两题都高于平均（正×正）或都低于平均（−×−）→ 正数；一高一低 → 负数
4. 公式：

$$r = \frac{\text{偏差乘积之和}}{\sqrt{x\text{ 偏差平方和} \times y\text{ 偏差平方和}}} = \frac{9}{\sqrt{10 \times 10}} = 0.9$$

- 分子：判断方向（一起变 → 正）
- 分母：把结果压到 −1 到 1 之间，方便比较

---

### 指标二：Cronbach's alpha
**看几道题整体有多一致**，范围 0 到 1

**核心想法**：如果几题一起高、一起低，把它们**加起来的总分**会被拉得很开（高的更高、低的更低），总分的波动会**特别大**。

**方差（variance）** = 偏差平方的平均，衡量一组数散得多开
- x 的方差 = 10 / 4 = 2.5
- y 的方差 = 10 / 4 = 2.5
- **各题方差之和 = 5**

**总分的方差**

| 人 | 总分 x + y | 偏差（平均 6） | 偏差平方 |
|---|---|---|---|
| 甲 | 9 | 3 | 9 |
| 乙 | 9 | 3 | 9 |
| 丙 | 4 | −2 | 4 |
| 丁 | 2 | −4 | 16 |
| | | 合计 | 38 |

**总分方差 = 38 / 4 = 9.5**（比各题方差之和 5 大很多，被"一起变"放大了）

公式（k = 题数）：

$$\alpha = \frac{k}{k-1} \times \left(1 - \frac{\text{各题方差之和}}{\text{总分方差}}\right) = \frac{2}{1} \times \left(1 - \frac{5}{9.5}\right) = 0.95$$

| 情况 | 各题方差之和 ÷ 总分方差 | alpha |
|---|---|---|
| 题目一起变 | 很小（总分被放大） | 接近 1 |
| 题目没关系 | 接近 1（没有放大） | 接近 0 |

- k/(k−1) 是修正系数，让结果落在 0–1
- 一般 0.7 以上可以合成，0.8 以上不错

---

### Q12 的初步检查结果
- 两两相关：0.49–0.63
- alpha = 0.84
- ⚠️ 这两个数**还需要我自己在 EDA 时推导并用代码算出来**验证

### 合成步骤
| 步骤 | 做什么 | 为什么 |
|---|---|---|
| ① | 6（不知道）→ 空 | "不知道"不是自信程度，不能当成数字 6 |
| ② | 反过来：6 − 原分数（1→5，5→1） | 让数字越大 = 越自信，和 Y1 方向一致 |
| ③ | 四题取平均 `.mean(axis=1)` | 得到 1–5 的自信分数 |

### 关于"不知道"
- 52 人至少有一题答了不知道，但**没有人四题都不知道**
- `.mean()` 默认跳过空值，用这个人答了的那几题算平均，所以**一个人都不会丢**

### 新学的
- 表格可以直接做算术：`6 - 表`，每一格都算
- `.mean(axis=1)`：横着取平均，跳过空值
- `.describe()`：数字变量的摘要（count、mean、std、min、25%、50%、75%、max）

### 后面会用到
| 系数 | 在哪里用 | 用来做什么 |
|---|---|---|
| 相关系数 | EDA、回归之前 | 看 X 和 Y 的关系；检查 X 之间是否太相关（多重共线性） |
| Cronbach's alpha | EDA、报告 Methods | 证明四题合成一个分数是合理的 |

In [140]:
v2b_path = '/content/drive/MyDrive/ai_project/sample_clean_v2b.csv'
sample.to_csv(v2b_path, index=False)
pd.read_csv(v2b_path).shape

(3515, 578)

In [141]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

path_3='/content/drive/MyDrive/ai_project/sample_clean_v2b.csv'
sample=pd.read_csv(path_3)
sample.shape


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(3515, 578)

In [142]:
q12_cols = ['Q12A', 'Q12B', 'Q12C', 'Q12D']

# 1. 提取四列，并将 6（不知道）替换为 NaN
q12 = sample[q12_cols].replace({6: np.nan})
print(q12.isna().sum())#各题"不知道=6"的人数

# 2. 反转分数：让数字越大越自信（1 变 5，2 变 4，3 变 3，4 变 2，5 变 1）
q12=6-q12 #有规律，6-每个数字刚好就反过来
#sample[q12_cols].replace({1:5, 2:4, 4:2, 5:1, 6:np.nan})相同效果
#.map() 都是对一列，现在是是一张表4列，replace更合适
print(q12.head()

)# 3. 计算四道题每个人平均分
sample['online_conf']=q12.mean(axis=1)

# 4. 查看描述性统计
sample['online_conf'].describe()#把3515人所有人分数相加再平均一次得到最终总平均分

Q12A     7
Q12B    26
Q12C    19
Q12D    10
dtype: int64
   Q12A  Q12B  Q12C  Q12D
0   3.0   NaN   5.0   5.0
1   3.0   4.0   2.0   4.0
2   2.0   4.0   4.0   4.0
3   2.0   2.0   2.0   2.0
4   5.0   5.0   5.0   5.0


,online_conf
count,3515.000000
mean,4.077264
std,0.699420
min,1.000000
25%,3.750000
50%,4.250000
75%,4.500000
max,5.000000


##A7 尝试新网站（最后一个变量）

## A7 尝试新网站：编码 4 并进 1

### 原始选项
| 编码 | 内容 | 人数 |
|---|---|---|
| 1 | 只用以前用过的网站/App | 1,406 |
| 2 | 试了一两个新的 | 1,467 |
| 3 | 试了很多新的 | 504 |
| 4 | 上个月没上网 | 47 |
| 5 | 不知道 | 91 |

### 处理
| 编码 | 处理 | 理由 |
|---|---|---|
| 4（47 人） | **并进 1**，共 1,453 人 | 够 30，规则上不必处理；但意思接近：上个月没上网 = 没有尝试新东西 |
| 5 不知道（91 人） | **单独一类**保留 | 够 30 |

### 新列 `new_sites_c`
| 编码 | 内容 | 人数 |
|---|---|---|
| 1 | 只用以前用过的 / 上个月没上网 | 1,453 |
| 2 | 试了一两个新的 | 1,467 |
| 3 | 试了很多新的 | 504 |
| 5 | 不知道 | 91 |

### 当数字还是当类别？
- 只看 1–3：有顺序，可以当数字
- 但保留了"不知道"：它不在这个顺序上，所以整个变量只能当**类别**
- 规律：一个变量里**所有的值**都能排成一个顺序，才能当数字；只要有一个排不进去，就只能当类别
- 折中做法（暂不采用）：1–3 当数字，另加一列"是不是不知道"的 0/1 标记（类似 `income_dk`）

In [143]:
sample['new_sites_c']=sample['A7'].replace({4:1})
sample['new_sites_c'].value_counts(dropna=False).sort_index()

,count
new_sites_c,
1.0,1453
2.0,1467
3.0,504
5.0,91


##删 19 人，存最终版

*   S4 性别 = 3（非二元）或 5（不愿说）：10 人

*   SEG = 5（不知道）：9 人




In [144]:
to_drop=sample['S4'].isin([3,5])|(sample['SEG']==5)
to_drop.sum()

np.int64(19)

In [145]:
sample=sample[~to_drop]
sample.shape

(3496, 580)

In [146]:
sample=sample.copy()

## pandas 常用操作总结：选东西、变空值、isin

### 一、`sample[...]`：方括号里放什么，就选什么

| 方括号里放 | 例子 | 拿到什么 |
|---|---|---|
| 一个列名 | `sample['A6']` | 一列 |
| 一个列名 list | `sample[['A3A', 'A3B']]` 或 `sample[a3_cols]` | 几列（一张小表） |
| 一列 True/False | `sample[sample['SUMAI'] == 1]` | 几行：True 的人留下，所有列都在 |

**例子（4 个人）**

| 行名 | SUMAI | A6 |
|---|---|---|
| 1 | 1 | 3 |
| 7 | 2 | 5 |
| 9 | 1 | 2 |
| 13 | 3 | 4 |

- `sample['A6']` → 3, 5, 2, 4
- `sample[sample['SUMAI'] == 1]` → 只留第 1 和第 9 行

**⚠️ 放在等号左边，意思就变了**

| 位置 | 例子 | 意思 |
|---|---|---|
| 等号右边 / 单独一行 | `x = sample[条件]` | 拿出来看或存到别处，sample 不变 |
| 等号左边 | `sample[条件] = np.nan` | 改 sample 本身：选中的行**整行**变空 ❌ |

---

### 二、让值变成 NaN 的方法

| 方法 | 例子 | 什么时候用 |
|---|---|---|
| `.replace()` 换成 `np.nan` | `sample[q12_cols].replace({6: np.nan})` | 某个**编码**要变空（6 = 不知道） |
| `.loc[行, 列] = np.nan` | `sample.loc[no_aiusers, 'Y2'] = np.nan` | 某些**人**的某**一列**要变空 |
| `.map()` 故意漏掉 | `sample['A6'].map({1:3, 2:2, 3:1, 4:0})` | 字典里没写的值自动变空（A6 = 5） |

- 相反的操作：`.fillna()` 把空值填上（A4_c、A5_c 填 0）
- `.loc` 指定了列，所以只改那一列；`sample[条件] = np.nan` 没指定列，所以整行都改了

---

### 三、`.isin()`：在不在这个 list 里

```python
sample['S4'].isin([3, 5])
```

| 行名 | S4 | `.isin([3, 5])` |
|---|---|---|
| 1 | 1 男 | False |
| 7 | 2 女 | False |
| 9 | 3 非二元 | True |
| 13 | 5 不愿说 | True |

- 等于 `(sample['S4'] == 3) | (sample['S4'] == 5)`，但更短
- 选项越多越省事：`.isin([1, 3, 5, 7, 9])`
- ⚠️ 不能写 `sample['S4'] == 3|5`：Python 会先算 `3|5`（= 7），变成 `== 7`

**结果是一列 True/False，可以：**
- 放进 `sample[...]` 筛人
- 放进 `.loc[..., 列]` 改值
- 用 `.sum()` 数有几个 True
- 存成变量

---

### 四、符号总结

| 符号 | 意思 |
|---|---|
| `==` | 等于 |
| `!=` | 不等于 |
| `&` | 并且 |
| `\|` | 或者 |
| `~` | 反过来（not） |

---

### 五、删 19 人的写法

```python
to_drop = sample['S4'].isin([3, 5]) | (sample['SEG'] == 5)   # 谁要删（一列 True/False）
to_drop.sum()                                                 # 应该是 19
```

- 这一步只是做标记，sample 还没变
- 下一步：`sample[~to_drop]` 留下其他人

In [147]:
sample['Y1'].notna().sum()

np.int64(3161)

In [148]:
sample['Y2'].notna().sum()

np.int64(2047)

In [149]:
v3_path='/content/drive/MyDrive/ai_project/sample_clean_v3.csv'
sample.to_csv(v3_path, index=False)
pd.read_csv(v3_path).shape


(3496, 580)